# Module 2.4: Procedural Memory

The agent now knows **what is true** (semantic memory) and **what happened**
(episodic memory). But it doesn't know **how to do things**.

Sarah asks: *"Book me an international flight to London."* The agent knows
she prefers United (semantic) and went to London last year (episodic). But
it doesn't know the 9-step international booking procedure — check visa,
verify insurance, enforce the 14-day advance booking rule, validate budget
against her Senior level.

Without a procedure, the agent guesses — and violates company policy.

| Chat History (2.1) | Episodic (2.2) | Semantic (2.3) | **Procedural (2.4)** |
|---|---|---|---|
| Remembers the thread | Stores what happened | Stores what is true | Stores **how to act** |
| Forgets between sessions | Recalls past events | Connects facts | Selects & follows workflows |
| Can't personalize | Personalizes from history | Reasons across knowledge | **Enforces policy & process** |

## What is Procedural Memory?

In cognitive science, procedural memory is *"how to ride a bike"* — learned
behaviors executed without conscious thought. For AI agents, it's the
multi-step workflows the agent selects and follows based on context.

There are three levels of sophistication:

| Level | Approach | How it works |
|---|---|---|
| **Static** | SkillsProvider | Developer authors SKILL.md files, agent loads by name |
| **Retrieved** | RAG (Azure AI Search) | Agent discovers procedures from indexed documents |
| **Learned** | Reflection (Cosmos DB) | Agent improves procedures from its own failures |

This notebook implements all three progressively — each solving a limitation
of the previous.

## Prerequisites

- Everything from prior modules (Azure AI Foundry, `.env`)
- **Azure AI Search** resource (free tier works)
  → Follow [steps/03_setup_search.md](steps/03_setup_search.md) if needed
- **Cosmos DB** (same database as Module 2.2)
- Add to your `.env`:
  ```
  SEARCH_ENDPOINT=https://your-search.search.windows.net
  SEARCH_KEY=your-admin-key
  ```

In [ ]:
%pip install -q -r ../../requirements.txt

In [ ]:
import sys, os, json
import sniffio
from pathlib import Path
from datetime import datetime, timezone

sys.path.insert(0, "./..")
sniffio.current_async_library_cvar.set("asyncio")

from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from shared.travel_agent import (
    create_client, SYSTEM_PROMPT,
    search_flights, search_hotels, get_travel_policy,
)

load_dotenv("./../.env", override=True)
client, credential = create_client("./../.env")
print("Foundry client ready")

In [ ]:
from agent_framework import tool

# In-memory stubs for episodic + semantic (focus stays on procedural)
user_preferences = {
    "E001": {"airline": "United", "hotel": "Marriott", "seat": "aisle",
             "level": "Senior", "dietary": "vegetarian"}
}

@tool
async def recall_user_preferences(user_id: str) -> str:
    """Recall known preferences for a user from semantic memory."""
    prefs = user_preferences.get(user_id, {})
    if not prefs:
        return f"No preferences found for {user_id}"
    return json.dumps(prefs, indent=2)

print("Memory stubs ready (in-memory preferences for Sarah/E001)")

## The Failure: No Procedure

Let's see what happens when the agent has facts but no workflow.
Sarah asks to book a $3,000 business class flight to London.

The agent has her preferences — but no procedure telling it to check
budget limits, verify visa status, or require travel insurance.

In [ ]:
from agent_framework import Agent, AgentSession

# V1: Agent has NO access to policies and NO procedure.
# It only knows how to search flights/hotels and recall preferences.
# Without a procedure, it will just find flights without checking policy.
V1_PROMPT = """You are a corporate travel assistant. Help employees book travel.
Use the tools to search for options. Be action-oriented — search immediately
with whatever info you have. Don't ask unnecessary questions. The user said
what they want — find it."""

v1_agent = Agent(
    client=client,
    name="TravelAssistant",
    instructions=V1_PROMPT,
    tools=[search_flights, search_hotels, recall_user_preferences],
)

async def demo_v1():
    session = AgentSession()
    msg = ("I'm Sarah Chen (E001), Senior Engineer based in New York. Book me a business class "
           "flight to London next week. Budget doesn't matter — find the best.")
    print(f"Sarah: {msg}")
    r = await v1_agent.run(msg, session=session)
    print(f"Agent: {r.text}")

await demo_v1()

### What Went Wrong

The agent searched for flights and is ready to book business class — no
questions asked about policy. It **violated company policy** on multiple fronts:

- Senior level max flight cost: **$800** — business class is $3,000+ ❌
- International trips require **travel insurance** — never mentioned ❌
- International trips require **14-day advance booking** — "next week" violates this ❌
- **Visa check** for UK entry — never asked ❌

The agent has the right facts (prefers United, aisle seat) but no
**procedure** to enforce policy constraints. Without `get_travel_policy`
and a step-by-step workflow, it can't know *when* or *how* to apply rules.

It needs a checklist — and the intelligence to select the right one.

---
## Approach 1: SkillsProvider (Static Procedures)

The simplest fix: give the agent a **checklist**. The Microsoft Agent
Framework provides `SkillsProvider` — it discovers SKILL.md files from
a folder and exposes them as tools:

- `load_skill(name)` — loads a specific procedure by name
- `read_skill_resource(name, resource)` — reads bundled data (budget limits, visa checklist)

```
skills/
  domestic-booking/
    SKILL.md              ← 6-step domestic procedure
    budget-limits.json    ← budget rules by level
  international-booking/
    SKILL.md              ← 9-step international procedure
    budget-limits.json
    visa-checklist.md     ← visa requirements
```

Each SKILL.md has YAML frontmatter (`name`, `description`) so the
framework can advertise it. The agent decides which skill to load
based on the user's request.

In [ ]:
from agent_framework import SkillsProvider

procedures = SkillsProvider.from_paths("skills")

# Load skills metadata to display what was discovered
skills = await procedures._source.get_skills()
print("SkillsProvider discovered:\n")
for skill in skills:
    print(f"  [{skill.frontmatter.name}] {skill.frontmatter.description}")
print(f"\nTools added: {procedures.LOAD_SKILL_TOOL_NAME}, {procedures.READ_SKILL_RESOURCE_TOOL_NAME}")

In [ ]:
from agent_framework import ToolApprovalMiddleware

V2_PROMPT = SYSTEM_PROMPT + "\n\n" + """You have procedural skills.
BEFORE booking any travel:
1. Determine if the trip is domestic or international
2. Call load_skill("domestic-booking") or load_skill("international-booking")
3. Follow the loaded procedure step by step
4. Use read_skill_resource() to get budget limits and visa checklists
5. NEVER skip a step — each exists for a policy reason
Always use user_id 'E001' for Sarah Chen."""

# Auto-approve load_skill / read_skill_resource so the agent doesn't pause for confirmation
approval = ToolApprovalMiddleware(
    auto_approval_rules=[SkillsProvider.all_tools_auto_approval_rule]
)

v2_agent = Agent(
    client=client, name="TravelAssistant",
    instructions=V2_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy,
           recall_user_preferences],
    context_providers=[procedures],
    middleware=[approval],
)
print("V2 agent ready (SkillsProvider + auto-approval)")

In [ ]:
async def demo_v2():
    session = AgentSession()
    msg = ("I'm Sarah Chen (E001), Senior Engineer based in New York. Book me a business class "
           "flight to London next week. Budget doesn't matter — find the best.")
    print(f"Sarah: {msg}")
    r = await v2_agent.run(msg, session=session)
    print(f"Agent: {r.text}")

await demo_v2()

### What Changed

The agent now **follows a procedure**: loads the international booking
skill, checks budget limits against Senior level ($800 max flight),
flags business class as policy-violating, mentions visa and insurance
requirements.

Same tools, same facts — different behavior. The procedure made the
difference.

**But there's a limitation**: every procedure must be hand-authored by
a developer as a SKILL.md file. The agent finds skills by exact name
(`load_skill("international-booking")`), not by understanding the
request. What if we had hundreds of procedures?

---
## Approach 2: RAG-Based Procedure Retrieval

What if the agent could **discover** procedures from company documents
instead of loading them by name?

We index procedure documents in **Azure AI Search**. The agent queries
by task intent — "how to book international travel" — and gets the
relevant procedure back. No developer authoring needed. Policy teams
update documents in the index; the agent discovers changes automatically.

| | SkillsProvider | RAG (Azure AI Search) |
|---|---|---|
| **Discovery** | By exact name | By semantic search |
| **Authoring** | Developer writes SKILL.md | Policy team updates docs |
| **Updates** | Edit file, commit, redeploy | Update index, instant |
| **Scale** | Dozens of skills | Hundreds of procedures |

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchFieldDataType,
)
from azure.core.credentials import AzureKeyCredential

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"]
SEARCH_KEY = os.environ["AZURE_SEARCH_KEY"]
INDEX_NAME = "travel-procedures"

search_cred = AzureKeyCredential(SEARCH_KEY)
index_client = SearchIndexClient(SEARCH_ENDPOINT, search_cred)
print(f"Azure AI Search: {SEARCH_ENDPOINT}")

In [ ]:
# Create the search index
index = SearchIndex(
    name=INDEX_NAME,
    fields=[
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="title", type=SearchFieldDataType.String),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="category", type=SearchFieldDataType.String, filterable=True),
    ],
)
index_client.create_or_update_index(index)
print(f"Index '{INDEX_NAME}' ready")

In [ ]:
# Index procedure documents from our SKILL.md files + travel policies
domestic_skill = Path("skills/domestic-booking/SKILL.md").read_text(encoding="utf-8")
intl_skill = Path("skills/international-booking/SKILL.md").read_text(encoding="utf-8")
visa_doc = Path("skills/international-booking/visa-checklist.md").read_text(encoding="utf-8")
policies = json.loads(Path("./../data/travel_policies.json").read_text(encoding="utf-8"))

documents = [
    {"id": "domestic-booking", "title": "Domestic Booking Procedure",
     "content": domestic_skill, "category": "booking"},
    {"id": "international-booking", "title": "International Booking Procedure",
     "content": intl_skill, "category": "booking"},
    {"id": "visa-checklist", "title": "Visa Requirements Checklist",
     "content": visa_doc, "category": "compliance"},
    {"id": "budget-policy", "title": "Travel Budget Policy by Employee Level",
     "content": json.dumps(policies, indent=2), "category": "policy"},
]

sc = SearchClient(SEARCH_ENDPOINT, INDEX_NAME, search_cred)
result = sc.upload_documents(documents)
print(f"Indexed {len(result)} procedure documents")

In [ ]:
@tool
async def search_procedures(task_description: str) -> str:
    """Search for relevant procedures and policies for a given task."""
    results = sc.search(search_text=task_description, top=3)
    docs = []
    for r in results:
        docs.append(f"### {r['title']}\n{r['content']}")
    if not docs:
        return f"No procedures found for: {task_description}"
    return "\n\n---\n\n".join(docs)

print("search_procedures tool ready")

In [ ]:
V3_PROMPT = SYSTEM_PROMPT + "\n\n" + """You have access to a procedure search.
BEFORE performing any multi-step task (booking, changes, cancellations):
1. Call search_procedures() with a description of what you need to do
2. Read the returned procedure carefully
3. Follow it step by step — do NOT skip steps
4. If the procedure mentions budget limits or policy rules, enforce them
Always use user_id 'E001' for Sarah Chen."""

v3_agent = Agent(
    client=client, name="TravelAssistant",
    instructions=V3_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy,
           recall_user_preferences, search_procedures],
)
print("V3 agent ready (RAG-based procedures)")

In [ ]:
async def demo_v3():
    session = AgentSession()
    msg = ("I'm Sarah Chen (E001), Senior Engineer based in New York. Book me a business class "
           "flight to London next week. Budget doesn't matter — find the best.")
    print(f"Sarah: {msg}")
    r = await v3_agent.run(msg, session=session)
    print(f"Agent: {r.text}")

await demo_v3()

### SkillsProvider vs RAG

The agent discovered the same international booking procedure — but
through **semantic search**, not by exact name. It queried "international
flight booking" and got the right checklist back.

RAG advantages:
- Policy teams update documents → agent discovers changes instantly
- Scales to hundreds of procedures without code changes
- Semantic matching finds the right procedure even with fuzzy queries

**But both approaches share a limitation**: procedures are static.
When the agent follows a procedure and still fails — because the
procedure is incomplete or the situation is novel — it has no way
to learn from the failure.

---
## Approach 3: Reflection-Based Learning

The agent followed the international procedure perfectly. But it
missed a nuance: Sarah's connecting flight routes through Dubai,
which requires a **transit visa** — something the base procedure
doesn't cover.

How does the agent learn? **Reflection**:
1. Agent completes a task → observes the outcome
2. Reflects: *"What went wrong? What should I do differently?"*
3. Stores the lesson in Cosmos DB (`procedural-reflections` container)
4. Next time: retrieves procedure from RAG + relevant past reflections

The procedure improves through experience — without human rewriting.

In [ ]:
from azure.cosmos import CosmosClient

COSMOS_ENDPOINT = os.environ["COSMOS_ENDPOINT"]
cosmos = CosmosClient(COSMOS_ENDPOINT, credential=credential)
db = cosmos.get_database_client("travel-memory")

# Create or get the reflections container
try:
    reflections_container = db.create_container(
        id="procedural-reflections", partition_key={"paths": ["/task_type"]}
    )
except Exception:
    reflections_container = db.get_container_client("procedural-reflections")

print(f"Cosmos DB: reflections container ready")

In [ ]:
from uuid import uuid4

@tool
async def store_reflection(task_type: str, lesson: str) -> str:
    """Store a lesson learned after completing or failing a task."""
    doc = {
        "id": str(uuid4()),
        "task_type": task_type,
        "lesson": lesson,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    reflections_container.upsert_item(doc)
    return f"Stored reflection for '{task_type}': {lesson}"

print("store_reflection tool ready")

In [ ]:
@tool
async def recall_reflections(task_type: str) -> str:
    """Recall past lessons learned for a type of task."""
    query = "SELECT * FROM c WHERE c.task_type = @tt ORDER BY c.timestamp DESC"
    items = list(reflections_container.query_items(
        query=query, parameters=[{"name": "@tt", "value": task_type}],
        enable_cross_partition_query=False,
    ))
    if not items:
        return f"No past reflections for task type: {task_type}"
    lessons = [f"- {item['lesson']} ({item['timestamp'][:10]})" for item in items[:5]]
    return f"Past lessons for '{task_type}':\n" + "\n".join(lessons)

print("recall_reflections tool ready")

In [ ]:
V4_PROMPT = SYSTEM_PROMPT + "\n\n" + """You have procedures + learning.
BEFORE any multi-step task:
1. Call search_procedures() to get the relevant procedure
2. Call recall_reflections() to get lessons from past attempts
3. Combine both — follow the procedure but apply the lessons
AFTER completing or failing a task:
4. Reflect: what worked? What was missing from the procedure?
5. Call store_reflection() with the lesson for future use
Always use user_id 'E001' for Sarah Chen."""

v4_agent = Agent(
    client=client, name="TravelAssistant",
    instructions=V4_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy,
           recall_user_preferences, search_procedures,
           store_reflection, recall_reflections],
)
print("V4 agent ready (RAG + Reflection)")

In [ ]:
async def demo_v4_learn():
    session = AgentSession()
    turns = [
        ("I'm Sarah Chen (E001) based in New York. Book me a flight to London. I'll connect "
         "through Dubai on the way."),
        ("Oh wait — I just found out I need a transit visa for Dubai! "
         "The procedure didn't mention that. Can you note that for next time?"),
    ]
    for msg in turns:
        print(f"Sarah: {msg}")
        r = await v4_agent.run(msg, session=session)
        print(f"Agent: {r.text}\n")

await demo_v4_learn()

In [ ]:
# Verify the reflection was stored
items = list(reflections_container.query_items(
    query="SELECT * FROM c", enable_cross_partition_query=True
))
print(f"Reflections stored: {len(items)}\n")
for item in items:
    print(f"  [{item['task_type']}] {item['lesson']}")

In [ ]:
async def demo_v4_apply():
    session = AgentSession()  # fresh session — no chat history
    msg = ("I'm Sarah Chen (E001) based in New York. I need to fly to London again, "
           "probably connecting through Dubai. What do I need?")
    print(f"Sarah: {msg}")
    r = await v4_agent.run(msg, session=session)
    print(f"Agent: {r.text}")

await demo_v4_apply()

### Key Insight: Procedures That Improve

The agent retrieved the same international booking procedure from RAG —
but this time it also retrieved its own past reflection about Dubai
transit visas. It **proactively** warned Sarah, even though the base
procedure doesn't mention it.

The procedure didn't change. The agent's *application* of it improved
because it remembered what went wrong last time. This is the core of
procedural learning:

```
RAG Procedure (static) + Past Reflections (learned) = Improved Behavior
```

---
## Comparison: Three Approaches

| Dimension | SkillsProvider | RAG | RAG + Reflection |
|---|---|---|---|
| **Discovery** | By exact name | By semantic search | Semantic + learned lessons |
| **Authoring** | Developer writes SKILL.md | Policy team updates docs | Auto-generated from experience |
| **Improvement** | Manual (edit file) | Update index | Automatic (learn from failures) |
| **Auditability** | Git history | Search index versions | Reflection store + timestamps |
| **Complexity** | Low | Medium | Medium-High |
| **Best for** | Small teams, few procedures | Large orgs, many policies | High-stakes tasks, novel situations |

## What Procedural Memory Still Cannot Do

We used RAG to retrieve procedures — and it worked. But RAG and memory
are **not the same thing**:

| | Memory | RAG |
|---|---|---|
| **Source** | Agent's own experience | External documents |
| **Lifecycle** | Created by agent interactions | Curated by humans |
| **Personalized** | Yes (per user/session) | No (shared knowledge) |
| **Example** | "Sarah prefers United" | "International trips need insurance" |

The next module explores this distinction: **where does memory end and
RAG begin?** And how should they work together?

### Architecture So Far

```mermaid
flowchart LR
  U[User] --> A[Agent]
  A --> T[Travel Tools]
  A --> CH[(Chat History<br/>Cosmos DB)]
  A --> EM[(Episodic Memory<br/>Cosmos DB)]
  A --> SM[(Semantic Memory<br/>Neo4j Graph)]
  A --> PM[(Procedural Memory<br/>Skills + RAG + Reflections)]
  PM -->|procedures + lessons| A
```

**Next**: Module 2.5 — Memory vs RAG.

In [ ]:
# # Cleanup
# print("Module 2.4 complete.")
# print("Resources used: Azure AI Search index, Cosmos DB container")
# print("To clean up: delete 'travel-procedures' index and 'procedural-reflections' container")